# 中間表現の動作の説明

このノートブックでは、中間表現（IR, Intermediate Representation）を `qret` コマンドを使って読み取る手順を確認します。
IR は Quration ライブラリの中心であるため、構造を把握しておくことでライブラリの理解が深まります。
本章では以下の 3 つのサブコマンドの使い方を確認しつつ、中間表現について説明します。

- `print` :命令列をプリントする。
- `diagram` :量子回路に関する様々な図を作成する。
- `simulate` :量子回路をシミュレートする。

本章で扱う対象は `quration-core/examples/data/circuit/add_craig_5.json` で、以下では `example.json` として扱います。

本章で到達する目標は次のとおりです。

- `qret print` を用いて、IR の関数単位・命令単位の構造を読み解けるようにする。
- `FunctionCall` と `BasicBlock` の位置関係を追跡し、分岐や呼び出しの挙動を説明できるようにする。
- `diagram` の各種図と `simulate` の結果を照合し、構造理解と意味論確認を一貫して行えるようにする。

まずは実行環境と入力ファイルを準備します。
以下のコードでは、 `qret` の実行パスはビルド結果を前提にしていますが、環境に応じて適切な値をセットしてください。

In [ ]:
import pathlib
import os
import platform
import graphviz
from IPython.display import Code

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
example_path = project_root / "quration-core" / "examples" / "data" / "circuit" / "add_craig_5.json"

os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

`qret` が実行可能であることを確認します。

In [ ]:
!qret --version

入力 JSON を確認します。

In [ ]:
Code(filename=example_path, language="json")

## qret print

`print` は IR の命令列を確認するサブコマンドです。

この章では `print` について、次の観点を順番に確認します。

- モジュール全体像の把握（関数数・関数名・引数の有無）
- 関数内部での命令列の並び（`Call`、測定、分岐、基本命令）
- 必要に応じた展開深度（`-d`）で、呼び出し階層をどこまで展開するか

In [ ]:
!qret print --help

まず `print -s` で対象モジュールの要約を確認し、どの関数が含まれるかを把握します。

In [ ]:
!qret print -i {example_path} -s

次に `UncomputeTemporalAnd` 関数についての要約を `print -s` で確認します。

量子ビット数、基本ブロック数、命令数などを確認できます。

In [ ]:
!qret print -i {example_path} -s -f "UncomputeTemporalAnd"

### FunctionCall（関数呼び出し）の見方

`print` サブコマンドで具体的に回路の命令列を表示します。

In [ ]:
!qret print -i {example_path} -f "UncomputeTemporalAnd"

`print` の命令列で `Call` が現れた場合は、`FunctionCall` に相当します。
この表現が示すのは、制御フロー上の「呼び出し」だけではなく、
引数束縛と戻り関係を含めた構造の移動です。
読む際は、次の 2 点をセットで確認すると解釈しやすくなります。

- **呼び出し先関数（callee）**: どの関数へ移るのか
- **引数の対応（operate 等）**: 呼び出し側の演算対象がどのように渡されるのか

`AddCraig(5)` のように再利用関数が多い回路では、`Call` の連鎖を先に追うことで、
上位関数の意味を崩さずに全体像を掴みやすくなります。
ネストした呼び出し順が分かると、図上の CallGraph と命令列の対応を自然に合わせやすくなります。

### BasicBlock（基本ブロック）の見方

`FunctionCall` と `BasicBlock` は密接に関連しています。
`BasicBlock` は分岐やジャンプの境界で分割された最小実行単位として扱われます。
そのため、`Call` の流れだけでは見えにくい「どこで分岐し、どこで収束するか」を理解するには、`BasicBlock` 単位の見方が有効です。
`print` で命令列を読む際も、`BasicBlock` を軸に「到達可能ノード」を追うと、分岐条件の影響範囲を見失いにくくなります。

`TemporalAnd` は全体の筋が見えやすい一方、`UncomputeTemporalAnd` では `entry`、`then_0`、`if_cont_0` といった
基本ブロック名が直接登場します。
ここでブロック名と命令列を対応づけると、どの分岐条件でどのパスが選ばれるかまで、読解の幅が広がります。
同じ関数内でも、測定結果に応じて経路が分岐する局面がどこかを素早く見つけられるようになります。

In [ ]:
!qret print -i {example_path} -f "TemporalAnd"

続けて `AddCraig(5)` を見て、呼び出し元の大枠と内部呼び出しの展開深度（`-d`）が表示にどのように反映されるかを確認します。
まずは展開なしで全体の入口と呼び出し関係を確認し、次に `-d 2` で内部の展開を深めて追跡します。
再帰的に深くなるほど行数は増えますが、どこで再利用が起きているかが同時に見えるため、意図的に深度を上げる価値があります。

In [ ]:
!qret print -i {example_path} -f "AddCraig(5)"

次に `-d 2` を指定し、`FunctionCall` を 2 段階まで展開して読みます。
この出力から、ネストしている呼び出しがどこで終了するか、あるいは別の関数へ継承されるかを確認できます。

In [ ]:
!qret print -i {example_path} -f "AddCraig(5)" -d 2

この時点で、`FunctionCall` の連鎖と `BasicBlock` 構造をテキストで追跡できる状態になります。
次は同じ情報を図で確認し、分岐や呼び出しの経路を視覚的に固定します。

## qret diagram

`diagram` は IR についての情報を図で表示するためのサブコマンドです。
現在実装している図は以下の 4 つです。

- Control Flow Graph (CFG)
- Call Graph
- 回路図
- Compute Graph

In [ ]:
!qret diagram --help

### CFG（制御フロー図）

CFG は、`BasicBlock` 同士の遷移（分岐・継続）をノードとエッジで表現します。
`Function` 内の制御フローを先に固定化しておくと、`print` では追いづらい実行経路の重なりを整理しやすくなります。

図を読む観点は次のとおりです。

- エントリー（`entry`）と終了（`if` 系の継続）を先に確認する
- 分岐条件（測定結果に依存する経路）で、次にどのブロックへ進むかを追う
- 同一命令列でも到達可能ノードが異なるかを比較する

In [ ]:
!qret diagram -i {example_path} --function "UncomputeTemporalAnd" --graph-format "CFG" -o { output_dir / "tutorial_2_diagram_cfg.dot"}

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_cfg.dot")

### CallGraph

CallGraph は、`FunctionCall` の依存関係をノード・エッジで確認するための図です。
`AddCraig(5)` のような階層化が深い回路では、再利用される関数をこの図で把握すると、
後続の最適化や分解対象の優先度を決めやすくなります。

In [ ]:
!qret diagram -i {example_path} --function "AddCraig(5)" --graph-format "CallGraph" -o { output_dir / "tutorial_2_diagram_call_graph.dot"}

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_call_graph.dot")

呼び出し回数情報を付加すると、同一関数が何度呼び出されているかが見え、
反復コストの高いパスを発見しやすくなります。

In [ ]:
!qret diagram -i {example_path} --function "AddCraig(5)" --graph-format "CallGraph" -o { output_dir / "tutorial_2_diagram_call_graph.dot"} --display_num_calls

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_call_graph.dot")

### ComputeGraph

ComputeGraph は、命令同士のデータ依存を可視化する補助図です。
ここでは「どの qubit がどの操作から来て、どの命令で変化し、どこに寄与したか」を追うことができます。
`CFG` が制御フローの道筋を示すのに対し、`ComputeGraph` はデータ依存の見取り図として活用できます。

In [ ]:
!qret diagram -i {example_path} --function "TemporalAnd" --graph-format "ComputeGraph" -o { output_dir / "tutorial_2_diagram_compute_graph.dot"}

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_compute_graph.dot")

### LaTeX

LaTeX は、図をレポートや論文向けに取り込める形式で出力するための形式です。

[quantikz](https://ctan.org/pkg/quantikz) というパッケージの形式で出力します。

In [ ]:
!qret diagram -i {example_path} --function "AddCraig(5)" --graph-format "LaTeX" -o { output_dir / "tutorial_2_diagram_latex.tex"}

Code(filename=output_dir / "tutorial_2_diagram_latex.tex", language="tex")

## qret simulate

`simulate` は、IR をシミュレートするためのサブコマンドです。

どのような方法で量子状態を保持してシミュレートするかということを `--state` という引数で指定できます。
現在サポートしているオプションは次の 2 つです。

- `Toffoli` : `(係数, 計算基底)` のリストとして量子状態を表現します。
    - `--max_superpositions` という引数で、このリストの長さの上限を設定します。デフォルトは 1 です。
    - 加算回路といった古典回路のシミュレートに適しています。
- `FullQuantum` : 量子ビット数を $n$ として $2^n$ のベクトルと量子状態を表現します。
    - 量子ビット数が少ない回路のシミュレートに適しています。

In [ ]:
!qret simulate --help

In [ ]:
!qret simulate -i {example_path} -f "TemporalAnd" -s Toffoli --max_superpositions 2 --init_state "011"

In [ ]:
!qret simulate -i {example_path} -f "TemporalAnd" -s FullQuantum --init_state "011"

最後に `AddCraig(5)` を `Toffoli` で実行し、加算が行われていることを確認します。

以下のコマンドで `dst` の量子ビットを `01100` で初期化し、 `src` の量子ビットを `11010` 初期化して in-place の加算を行います。
実行結果としては、 `dst` の量子ビットだけが `10001` に変化することが期待されます。

In [ ]:
!qret simulate -i {example_path} -f "AddCraig(5)" -s Toffoli --max_superpositions 16 --init_state "0110011010000"

### （発展）古典的確率分布に従った量子回路の記述

Quration の中間表現では `DISCRETE_DISTRIBUTION` という命令があり、
ランダムな整数キーを classical レジスタ列として実体化し、
続く `SWITCH` で整数キーに対応する経路を選ぶ構成を作れます。
この例では、`q0, q1, q2` のいずれかへ 1/3 の確率で `X` を適用する分岐回路を構成します。

まず `DISCRETE_DISTRIBUTION` の挙動です。
- `weights` に応じて、`0` から `weights.size()-1` の整数値を 1 つ決定します。
- 決定した整数を、`registers` で与えたレジスタ列にビット列として書き込みます。
- この書き込みは LSB-first です。`@r0` が下位ビット（2^0）、`@r1` が次ビット（2^1）。

次に `SWITCH` のキー解釈です。
- `SWITCH` は `registers` レジスタ列を上位から下位ではなく、
  `registers[0]` を最下位ビットとして `BoolArrayAsInt` で整数化します。
- ここでの値は `0` 〜 `3` を扱う 2 ビット整数です。

具体的な対応は次のとおりです。
- `@r0=0, @r1=0` のとき: インデックス 0 → `case_X0`
- `@r0=1, @r1=0` のとき: インデックス 1 → `case_X1`
- `@r0=0, @r1=1` のとき: インデックス 2 → `case_X2`
- `@r0=1, @r1=1` のとき: インデックス 3 は `default`（それ以外の値は `return`）

各 case は 1 つの基本ブロックで、対応する 1 量子ビットへの `X` 実行後に `return` へ分岐します。
`weights = [1,1,1]` なので、`case_X0`,`case_X1`,`case_X2` はおおむね 1/3 ずつ選ばれます。
`index=3` はケース未定義なので `default` を通って `return` になります。
`simulate --sample_summary` では `100`,`010`,`001` がそれぞれほぼ等比率で観測され、
さらに文字列の並びは `q0 q1 q2`（LSB-first）である点を踏まえて `X` 適用先と対応づけられます。

まずサンプル入力ファイルを `dist_json` として固定し、以降の確認で同一入力を使える状態にします。

In [ ]:
dist_json = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_2.json"

Code(
    filename=dist_json,
    language="json",
)

この `print` で `DiscreteDistribution` と `Switch` の命令列を確認し、`case` の対応表をテキストで把握します。

In [ ]:
!qret print -i {dist_json} -f "Tutorial2Function"

次に CFG を描画して、`entry` から `case_X0/X1/X2` と `default` の流れを図で確認します。

In [ ]:
!qret diagram -i {dist_json} --function "Tutorial2Function" --graph-format "CFG" -o { output_dir / "tutorial_2_diagram_cfg.dot"}

graphviz.Source.from_file(output_dir / "tutorial_2_diagram_cfg.dot")

最後に `simulate` でサンプル分布を確認し、`case` 3 つの観測確率が 1/3 付近になることを確認します。

In [ ]:
!qret simulate -i {dist_json} -f "Tutorial2Function" -s FullQuantum --init_state "000" -n 10000 --sample_summary

## まとめ

- まず `print -s` でモジュール規模と関数一覧を確認し、調査対象を確定する
- 次に `print -f` で `FunctionCall` を軸に命令列を確認し、必要に応じて `-d` で展開深度を上げる
- `BasicBlock` を中心に `diagram --graph-format CFG` で分岐フローを把握する
- `CallGraph` で関数呼び出し依存を確認し、`ComputeGraph` でデータ依存を補完する